# DISTILBERT TRAINING

In [1]:
import pandas as pd

# Monte Google Drive pour accéder aux fichiers
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## Data Loading

In [2]:
# Charger le dataset depuis Google Drive
# df = pd.read_csv('../../data/datasets/full_dataset.csv')
df = pd.read_csv('/content/drive/MyDrive/FAKE_NEWS_PROJECT/full_dataset.csv')
df.head()

,Unnamed: 0,text,label
0,18549,"DHAKA, (Reuters) - Bangladesh and Myanmar agr...",1
1,1133,18 percent of our land in our state right now ...,1
2,21643,Hahahahahahahaha. Hahahahahahahahahahahahahaha...,0
3,32595,"Omarosa Manigault, a senior staff member of Pr...",0
4,12687,"The port provides more than 297,000 jobs direc...",1


## Data Exploration and Preprocessing

In [3]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Labels: {df["label"].value_counts()}")

Shape: (46687, 3)
Columns: ['Unnamed: 0', 'text', 'label']
Labels: label
1    25694
0    20993
Name: count, dtype: int64


### Filtering Data by Text Length

To avoid bias towards long texts and align with a tweet-like length, we will filter the dataset to include only texts shorter than 1000 characters.

In [4]:
# Filter out texts longer than 1000 characters
original_num_rows = len(df)
df = df[df["text"].str.len() < 1000]
filtered_num_rows = len(df)

print(f"Original number of rows: {original_num_rows}")
print(f"Number of rows after filtering (text length < 1000): {filtered_num_rows}")
print(f"Number of rows removed: {original_num_rows - filtered_num_rows}")

Original number of rows: 46687
Number of rows after filtering (text length < 1000): 15770
Number of rows removed: 30917


### De-leaking the text (CRITICAL)

The ISOT "real" articles almost all start with a Reuters source signature
(e.g. `WASHINGTON (Reuters) -`) and a city dateline, while the "fake" ones are
sensational (lots of `!`). A model trained on the raw text learns to detect the
**Reuters byline**, not the truthfulness — ~99% holdout accuracy that collapses
on real-world text.

We strip those source tells before training. This is the **same `clean_text`**
used at inference time (`preprocessing.py`), so train and inference see the same
text distribution.

In [5]:
import re

# Identical to clean_text() in preprocessing.py (regex-only part, no spaCy needed
# in Colab). Keep these two in sync so training and inference see the same text.
def clean_text(text):
    """Cleans the input text by removing source signatures, URLs, mentions, numbers."""
    # Normalize curly/smart quotes, dashes, spaces
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace(" ", " ").replace("…", "...")
    # Normalize exclamation marks: ISOT "fake" texts use "!" ~8x more than "real",
    # so the model learned "!" = fake and a single "!" flips a neutral fact.
    # Collapse any run of "!" to a single "." so punctuation can't drive veracity.
    text = re.sub(r'!+', '.', text)
    # Remove source signatures at the start: "WASHINGTON (Reuters) -"  <-- the main leak
    text = re.sub(r'^[A-Z\s/,]+ \([A-Za-z\s]+\)\s*[-–—]\s*', '', text)
    # Remove photo attributions
    text = re.sub(r'(?i)(photo|image|featured? image|featured? sketch)\s*(by|via|courtesy of)\s*.{0,80}', '', text)
    # Remove call-to-action patterns
    text = re.sub(r'(?i)(read more|watch .{0,20}video|click here|subscribe|share this|sign up).*', '', text)
    # Remove URLs / mentions / numbers
    text = re.sub(r'https?://\S+|www\.\S+|\S+\.\w{2,}/\S*', '', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'[\$€£]?\d[\d,.]*', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply de-leaking and drop rows that became empty after cleaning
df["text"] = df["text"].astype(str).apply(clean_text)
df = df[df["text"].str.strip().astype(bool)].reset_index(drop=True)

# Sanity check: Reuters byline + "!" rate should now be near-zero
real = df[df["label"] == 1]["text"]
print(f"Rows after de-leaking: {len(df)}")
print(f"Real-class rows still starting with a (Reuters) byline: "
      f"{real.str.match(r'.{0,40}\\(Reuters\\)').mean():.2%}")
print(f"Rows still containing '!': {df['text'].str.contains('!', regex=False).mean():.2%}  (should be 0%)")


Rows after de-leaking: 15707
Real-class rows still starting with a (Reuters) byline: 0.00%
Rows still containing '!': 0.00%  (should be 0%)


### Balancing text length across classes (de-confounding)

Second leak, subtler than the Reuters byline: **text length correlates with the
label**. The "real" class is dominated by long Reuters articles, the "fake" class
by short claims:

| Length | real% |
| --- | --- |
| `< 20` chars | 8% |
| `20–40` | 34% |
| `40–60` | 50% |
| `400–1000` | 81% |

So the model learns *"long → real, short → fake"* and confidently calls a neutral
one-liner ("Good morning", "Macron arrived in Paris") **Fake** — purely because
it is short, not because of its content.

We remove that signal by **matching the length distribution across classes**:
within each length bucket we downsample the majority class to the minority count.
After this, length no longer tells the model anything, so short neutral text
lands near 0.5 (honest "I don't know") instead of a confident wrong "Fake".

In [6]:
import numpy as np

# Match the length distribution across classes so the model can't use length as
# a proxy for the label. We bucket by character length and, in each bucket,
# downsample the majority class to the minority count.
_len = df["text"].str.len()
bins = [0, 40, 80, 160, 320, 520, 10_000]
bucket = pd.cut(_len, bins=bins, labels=False, include_lowest=True)

before_med = df.groupby("label")["text"].apply(lambda s: int(s.str.len().median())).to_dict()

parts = []
for _, grp in df.groupby(bucket):
    real = grp[grp["label"] == 1]
    fake = grp[grp["label"] == 0]
    n = min(len(real), len(fake))
    if n == 0:
        continue
    parts.append(real.sample(n=n, random_state=42))
    parts.append(fake.sample(n=n, random_state=42))

df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

after_med = df.groupby("label")["text"].apply(lambda s: int(s.str.len().median())).to_dict()

print(f"Balanced dataset: {len(df)} rows")
print(df["label"].value_counts())
# The two medians should now be close -> length no longer separates the classes
print(f"Median length by label  before: {before_med}  ->  after: {after_med}")

Balanced dataset: 10956 rows
label
0    5478
1    5478
Name: count, dtype: int64
Median length by label  before: {0: 116, 1: 331}  ->  after: {0: 118, 1: 118}


## GPU Setup

In [7]:
import torch
# Verification du GPU
print(f"GPU disponible: {torch.cuda.is_available()}")
# Nom du GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
# VRAM
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

GPU disponible: True
GPU: Tesla T4
VRAM: 15.6 Go


## Tokenization

In [8]:
import transformers as tf

tokenizer = tf.AutoTokenizer.from_pretrained("distilbert-base-uncased")
ex = tokenizer("Hello World")
print(ex)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


In [9]:
print(tokenizer.decode([101]))
print(tokenizer.decode([102]))

[CLS]
[SEP]


"Hello World" c'est 2 mots mais on a 4 input_ids :
- 101 = [CLS] — marqueur de début de séquence.
- 102 = [SEP] — marqueur de fin de séquence.

### Tokenization example with padding - LIAR (short text)

In [10]:
# Premier text (text court)
example = df["text"].iloc[0]

In [11]:
tokens = tokenizer(example, max_length=512, truncation=True, padding="max_length")

print(f"Example: {example[:100]}...")
print(f"Tokens numbers: {sum(tokens["attention_mask"])}")
print(f"Total length (avec padding): {len(tokens["input_ids"])}")

Example: There is no disagreement that we need action by our government, a recovery plan that will help to ju...
Tokens numbers: 27
Total length (avec padding): 512


Le texte fait 119 vrais tokens. Mais avec max_length=512, le tokenizer a rajouté 393 tokens de padding (des zéros) pour atteindre 512. C'est à ça que sert l'attention_mask. Il informe le modèle : "les 119 premiers sont du vrai texte, le reste c'est du remplissage à ignorer."

### Tokenization example without padding - ISOT (long text)

In [12]:
# Récupère un text long
longest_id = df["text"].str.len().idxmax()
example = df["text"].iloc[longest_id]

In [13]:
tokens = tokenizer(example, max_length=512, truncation=True, padding="max_length")

print(f"Example: {example[:100]}...")
print(f"Nb tokens: {sum(tokens["attention_mask"])}")
print(f"Longueur totale (avec padding): {len(tokens["input_ids"])}")

Example: The left is quickly discovering that this is a very different crop of GOP Presidential candidates, w...
Nb tokens: 200
Longueur totale (avec padding): 512


512 tokens, zéro padding, ce texte a été tronqué. Tout ce qui dépasse 512 tokens est perdu.

## Data Splitting

Pour la baseline on a utlisé 80/20 (train/test). Pour DistilBERT, on a besoin d'un validation set afin gerer la perte pendant l'entraînement et éviter l'overfitting.

**Répartition classique : 80% train / 10% validation / 10% test**

In [14]:
from sklearn.model_selection import train_test_split

# Découpage -> entrainement (80%) & données de test temporaires (20%)
# 'stratify=df["label"]' asssure que la proportion de label est maintenue entre chaque sets
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], train_size = 0.8, random_state = 42, shuffle = True, stratify = df["label"])

# Découpage restant des données de test temporaires en validation (10%) et test (10%)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, train_size=0.5, random_state = 42 , shuffle = True, stratify = y_test)

print(f"Train: {len(X_train)} / Valid: {len(X_valid)} / Test: {len(X_test)}")
print(f"Train labels:\n{y_train.value_counts()}")

Train: 8764 / Valid: 1096 / Test: 1096
Train labels:
label
1    4382
0    4382
Name: count, dtype: int64


## Tokenisation du dataset

Le Trainer de Hugging Face ne travaille pas avec des DataFrames pandas. Il attend un objet Dataset de la librairie datasets. C'est un format optimisé pour le traitement par batch, le mapping de fonctions (comme la tokenisation), et le chargement en mémoire efficace.

In [15]:
from datasets import Dataset

# Création des datasets au format Hugging Face
train_dataset = Dataset.from_dict({"text": X_train.to_list(), "label": y_train.to_list()})
valid_dataset = Dataset.from_dict({"text": X_valid.to_list(), "label": y_valid.to_list()})
test_dataset = Dataset.from_dict({"text": X_test.to_list(), "label": y_test.to_list()})

# Fonction de tokenisation
def tokenize(batch):
  return tokenizer(batch["text"], max_length=128, truncation=True, padding="max_length")

# On Tokenize les 3 datasets
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/8764 [00:00<?, ? examples/s]

Map:   0%|          | 0/1096 [00:00<?, ? examples/s]

Map:   0%|          | 0/1096 [00:00<?, ? examples/s]

In [16]:
print(train_dataset)
print(train_dataset[0].keys())

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8764
})
dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


Le dataset contient maintenant les **5 colonnes** : text et label d'origine, plus input_ids, attention_mask et token_type_ids ajoutés par le tokenizer.

Le Trainer n'a besoin que de input_ids, attention_mask et label. La colonne text ne sert plus (DistilBERT travaille avec les tokens, pas le texte brut) et token_type_ids est inutile pour DistilBERT.

## Pre-trained model loading

DistilBert a été créé "deviner" un mot manquant dans une phrase.

Pour cela, il y a deux parties:
- le cerveau -> Les 6 couches d'attention qui comprennent le language. C'est grâce à ces 6 couches d'attention que le modèle apprend la grammaire, le sens des mots et les relations contextuelles.
- la tête "deviner les mots" -> C'est la couche à la fin qui prend la compréhension du cerveau et produit un mot.

Ici, en appliquant *AutoModelForSequenceClassification*, on garde le cerveau du modèle pré-entrainé et l'on se débarasse de la tête "deviner les mots" pour la remplacer par notre nouvelle tête "classifier en 2 classes (fake/real).

In [17]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # num_labels = 2 -> 2 classes (fake/real)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


On peut observer ce qu'il s'est passé. On voit que l'ancienne tête du modèle (status *UNEXPECTED*) a été remplacée par la nouvelle tête de classification (status *MISSING*).

UNEXPECTED -> se débarasse des composants de l'ancienne tête.
MISSING -> Ajoute les composants nécéssaire à la nouvelle tête de classification.

## Fine Tuning

In [18]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import f1_score, classification_report

def get_metrics(eval_prediction):
  logits, labels = eval_prediction # logits -> score brut en sortie du model / labels -> représente les vrais labels pour comparer aux scores bruts
  preds = np.argmax(logits, axis=-1) # on récupère l'index du score le plus élevé
  f1 = f1_score(labels, preds, average="weighted") # calcul du score F1
  return {"f1": f1} # retourne le score F1

# Parametre d'entrainement - Avec les hyperparamètres standard de BERT
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3, # Le nombre de fois que le model voit le dataset passer
    per_device_train_batch_size=16, # le nombre de text a traiter en même temps
    per_device_eval_batch_size=16, # idem mais pour le dataset d'évaliation
    learning_rate=2e-5, # Vitesse d'apprentissage -> Trop hau: le modele "oublie", / Trop bas : il n'apprend pas
    weight_decay=0.01, # Régularisation pour éviter l'overfitting : C'est l'équivalent du paramètre C dans logistic regression
    eval_strategy="epoch", # Evalue le modèle sur le set de validation après epoch
    save_strategy="epoch", # sauvegarde un checkpoint à la fin de chaque epoch
    load_best_model_at_end=True, # Garde le meilleur score (ici F1) des trois dernier epoch
    metric_for_best_model="f1", # le scrore qui determoinbe le meilleur modele
    fp16=True, # Réduit la VRAM utilisé (travail en 16 bits au lieu de 32 bits), accélère l'entrainement
    report_to="none", # Pas de logging
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    compute_metrics = get_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.485375,0.467133,0.731394
2,0.393713,0.507942,0.740458
3,0.311273,0.590029,0.725399


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1644, training_loss=0.3875217344929122, metrics={'train_runtime': 124.1193, 'train_samples_per_second': 211.829, 'train_steps_per_second': 13.245, 'total_flos': 870708211365888.0, 'train_loss': 0.3875217344929122, 'epoch': 3.0})

On observe entre chaque epoch une chute du *Training Loss* et cela est normal puisque le modèle apprend de mieux en mieux sur les données d'entrainement.

Cependant, la *Validation Loss* augmente entre l'epoch 2 et l'epoch 3. Cela signifie que le modèle devient meilleur sur les données qu'il connait, mais moins bon sur les données qu'il n'a jamais vues.

**C'est un signal d'OVERFITTING** -> Le modèle commence à mémoriser les exemples d'entraînement au lieu d'apprendre des règles générales.

C'est tout l'intérêt du paramètres *load_best_model_at_end=True*, le Trainer regarde quel epoch avait le meilleur F1 sur la validation et recharge celui ci. Sans ce paramètre, le modèle de l'epoch final aurait été gardé.

## Evaluation

In [19]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(y_test, preds, target_names=["Fake (0)", "Real (1)"]))

              precision    recall  f1-score   support

    Fake (0)       0.81      0.71      0.75       548
    Real (1)       0.74      0.83      0.78       548

    accuracy                           0.77      1096
   macro avg       0.77      0.77      0.77      1096
weighted avg       0.77      0.77      0.77      1096



## Comparaison avec le baseline

| Modèle | F1 | Precision | Recall |
| --- | --- | --- | --- |
| TF-IDF + Logistic Regression | 0.90 | 0.90 | 0.90 |
| DistilBERT (texte brut, **avec fuite**) | ~0.95 | ~0.95 | ~0.95 |
| DistilBERT (dé-fuité + ponctuation normalisée) | 0.82 | 0.83 | 0.83 |
| DistilBERT (**+ longueur équilibrée**) | **0.77** | **0.77** | **0.77** |

**Pourquoi le F1 descend de 0.95 → 0.82 → 0.77 ?**

À chaque étape on **retire une fuite**, et le score baisse parce qu'il devient
plus honnête. Trois corrections, toutes appliquées dans `clean_text` / au
dataset (train **et** inference) :

1. **Dé-fuitage de la source** — on retire la signature Reuters et la dateline.
   La version à 0.95 lisait *« (Reuters) »*, pas le contenu.
2. **Normalisation de la ponctuation** — les `!` (8× plus fréquents dans les
   « fake ») deviennent `.`. Avant, un simple `!` faisait basculer une phrase
   neutre de *Real* à *Fake*.
3. **Équilibrage de la longueur** — le « real » était surtout des longs articles
   Reuters, le « fake » des phrases courtes : le modèle apprenait *« long → real,
   court → fake »* et classait « Good morning » en *Fake* à 98%. On égalise la
   distribution de longueur par classe (médiane 331/116 → 118/118).

L'étape 3 réduit le dataset (15 707 → 10 956 lignes) : pour que la longueur ne
prédise plus le label, chaque tranche de longueur doit contenir autant de « real »
que de « fake », donc on écarte le surplus de longs articles « real ». C'est le
**prix de la suppression du biais**. Pour un modèle pré-entraîné, l'équilibre
compte plus que le volume — 8 764 exemples d'entraînement suffisent largement.

Le holdout tombe à **0.77**, attendu : le test est désormais équilibré (548/548)
et le modèle n'a plus aucun raccourci (ni source, ni `!`, ni longueur).

**La vraie preuve est ailleurs : sur le set hors-distribution fait main, le score
monte de 94% à 100% (16/16).** En perdant ses raccourcis, le modèle est devenu
*meilleur* sur du vrai texte court — l'inverse d'une régression.

> Leçon (pour le rapport) : un score de holdout élevé est souvent un signal de
> fuite de données, pas de performance. Un holdout qui baisse pendant qu'un set
> hors-distribution monte = on a remplacé des raccourcis par de la vraie capacité.

In [20]:
test_texts = [
    "The government has announced a new policy to combat climate change.",
    "The spokesman for the ministry issued a statement saying the government would seek to address the allegations raised by the parliamentary committee investigating the matter.",
    "You won't believe what this racist politician just revealed about his secret scheme. Watch the video and share this story before they try to hide the truth from liberal America."
]

from transformers import pipeline
clf = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)
# clf = pipeline("text-classification", model="../models/distilbert", tokenizer="../models/distilbert", device=0)
for text in test_texts:
    result = clf(text)
    print(f"{text[:80]}...")
    print(f"  -> {result}\n")

The government has announced a new policy to combat climate change....
  -> [{'label': 'LABEL_0', 'score': 0.546567440032959}]

The spokesman for the ministry issued a statement saying the government would se...
  -> [{'label': 'LABEL_1', 'score': 0.9948102235794067}]

You won't believe what this racist politician just revealed about his secret sch...
  -> [{'label': 'LABEL_0', 'score': 0.9956516623497009}]



## Évaluation hors distribution (OOD)

Le holdout reste de la même distribution (ISOT/LIAR). Pour mesurer la **vraie**
capacité du modèle, on l'évalue sur un petit jeu de phrases courtes, labellisées
à la main, qui ressemblent à de vrais posts / tweets — donc hors de la
distribution d'entraînement.

On y inclut la paire « avec / sans `!` » (Trump en Inde) pour vérifier que la
**normalisation de la ponctuation** (`!` → `.`) a supprimé la sur-sensibilité au
`!` : les deux phrases doivent donner exactement la même prédiction.

**Résultat obtenu :**
- OOD = **100% (16/16)** — contre 94% avant l'équilibrage de la longueur. Les
  phrases neutres courtes, qui partaient en *Fake* à cause de leur longueur, sont
  maintenant correctement classées.
- Paire « . / ! » : prédiction **identique** (LABEL_1 0.718 dans les deux cas) —
  le `!` ne fait plus basculer le résultat.

C'est le set OOD, pas le holdout, qui reflète l'usage réel : un modèle sans
raccourci marque moins sur son holdout mais **mieux sur du vrai texte**.

In [21]:
from sklearn.metrics import classification_report

# Petit jeu hors distribution, labellisé à la main.  label: 1 = Real, 0 = Fake.
# clean_text() est appliqué pour matcher exactement l'entrée vue à l'entraînement.
ood_samples = [
    # --- Real : factuel, sobre, vérifiable ---
    ("The central bank kept interest rates unchanged at its meeting on Thursday.", 1),
    ("Researchers published a study on coral reef recovery in the journal Nature.", 1),
    ("The mayor signed the new housing budget into effect this week.", 1),
    ("Heavy rainfall caused minor flooding in several coastal towns overnight.", 1),
    ("The company reported quarterly earnings slightly above analyst expectations.", 1),
    ("Officials confirmed the road will reopen after repairs are completed.", 1),
    ("A new train line connecting the two cities opened to passengers on Monday.", 1),
    ("The committee will review the proposal at its next scheduled session.", 1),
    # --- Fake : sensationnel, complot, appât émotionnel ---
    ("BREAKING: Scientists CONFIRM the moon landing was faked all along!!!", 0),
    ("You won't believe what doctors are HIDING about this miracle cure!", 0),
    ("SHOCKING: Secret elite group controls the weather to enslave us all.", 0),
    ("This one weird trick will make you rich overnight, banks HATE it!", 0),
    ("They don't want you to know the vaccine contains mind-control microchips.", 0),
    ("WAKE UP people!! The government is poisoning the water on purpose.", 0),
    ("A celebrity just exposed the entire system in this leaked video, share before deleted!", 0),
    ("Aliens have been living among us and the media refuses to report it.", 0),
]

# --- Paire ponctuation (test de la sur-sensibilité au "!") ---
punct_pair = [
    "Donald Trump arrived in India with his wife Melania.",
    "Donald Trump arrived in India with his wife Melania!",
]

texts  = [clean_text(t) for t, _ in ood_samples]
labels = [lbl for _, lbl in ood_samples]

# Le modèle n'a pas encore reçu id2label={0:Fake,1:Real} (fait à la sauvegarde),
# donc le pipeline `clf` renvoie 'LABEL_0' / 'LABEL_1' ici.
def to_idx(lbl):
    return 1 if lbl in ("Real", "LABEL_1") else 0

preds = []
print("=== Prédictions OOD ===")
for (raw, gold), clean in zip(ood_samples, texts):
    r = clf(clean)[0]
    p = to_idx(r["label"])
    preds.append(p)
    ok = "OK   " if p == gold else "WRONG"
    print(f"[{ok}] gold={gold} pred={p} ({r['score']:.2f})  {raw[:70]}")

acc = sum(int(p == g) for p, g in zip(preds, labels)) / len(labels)
print(f"\nOOD accuracy: {acc:.2%}  ({sum(int(p==g) for p,g in zip(preds,labels))}/{len(labels)})")
print(classification_report(labels, preds, target_names=["Fake (0)", "Real (1)"], zero_division=0))

print("=== Test ponctuation (le '!' doit PEU/PAS changer la prédiction) ===")
for t in punct_pair:
    r = clf(clean_text(t))[0]
    print(f"  {r['label']} ({r['score']:.3f})  <- {t!r}")


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== Prédictions OOD ===
[OK   ] gold=1 pred=1 (0.99)  The central bank kept interest rates unchanged at its meeting on Thurs
[OK   ] gold=1 pred=1 (0.54)  Researchers published a study on coral reef recovery in the journal Na
[OK   ] gold=1 pred=1 (0.66)  The mayor signed the new housing budget into effect this week.
[OK   ] gold=1 pred=1 (0.99)  Heavy rainfall caused minor flooding in several coastal towns overnigh
[OK   ] gold=1 pred=1 (0.94)  The company reported quarterly earnings slightly above analyst expecta
[OK   ] gold=1 pred=1 (0.98)  Officials confirmed the road will reopen after repairs are completed.
[OK   ] gold=1 pred=1 (0.99)  A new train line connecting the two cities opened to passengers on Mon
[OK   ] gold=1 pred=1 (0.65)  The committee will review the proposal at its next scheduled session.
[OK   ] gold=0 pred=0 (0.99)  BREAKING: Scientists CONFIRM the moon landing was faked all along!!!
[OK   ] gold=0 pred=0 (0.99)  You won't believe what doctors are HIDING about t

In [22]:
save_path = "/content/drive/MyDrive/FAKE_NEWS_PROJECT/distilbert_model"
model.config.id2label = {0: "Fake", 1: "Real"}
model.config.label2id = {"Fake": 0, "Real": 1}
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Modèle sauvegardé dans {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé dans /content/drive/MyDrive/FAKE_NEWS_PROJECT/distilbert_model
